# Research foundations: saved inputs to a research dataset

For researchers with saved OKX historical L2 and Bitfinex R0 recordings, this
walkthrough answers three questions:

1. What visible liquidity can I reconstruct, and which clocks are available?
2. How do separate saved inputs flow through admission into one reproducible dataset?
3. Which revisions and membership assertions were known at a decision cutoff?

The book exploration uses real saved inputs. The dataset walkthrough uses **small
synthetic trades**, because published LOB adapters are planned for MR 25. Everything
here runs offline, without credentials. If inputs are missing, first use
[market-data-capture.ipynb](market-data-capture.ipynb); see the
[operations guide](notebook-operations.md) for installation and pinned input paths.

Restart the kernel and **Run All in order**. Expand the collapsed setup cells only
when needed. These observations do not replace the [frozen P1 study](../docs/research/p1-baseline.md)
or establish coverage, fills or a strategy result.

## 1. Identify this execution

The installed package must match the checkout and pinned interpreter. The table
records code, configuration and notebook fingerprints for comparing later runs.

In [1]:
# standard library imports
import hashlib
import json
import os
import platform
import subprocess
import sys
from datetime import UTC, datetime, timedelta
from decimal import Decimal
from html import escape
from importlib.metadata import distributions, version
from pathlib import Path

# third party library imports
from IPython.display import HTML, display

# project library imports
import alpha_research
from alpha_research.data.live.config import CaptureConfig, CaptureRequest
from alpha_research.data.live.journal import CaptureJournal, CaptureRecord
from alpha_research.data.live.replay import CaptureReplay
from alpha_research.data.raw_objects import RawObject
from alpha_research.data.venues.bitfinex.book import BitfinexBook
from alpha_research.data.venues.okx.history_config import OKXArchiveRequest, OKXHistoryConfig
from alpha_research.data.venues.okx.history_store import OKXArchiveStore
from alpha_research.serialization import DomainCodec

root = Path.cwd().resolve()
if root.name == "examples":
    root = root.parent
assert (root / "configs/history/okx-btc-usdt.json").is_file(), (
    "Run from the repository root or examples/."
)
notebook_path = Path(
    os.environ.get("ALPHA_RESEARCH_NOTEBOOK_PATH", root / "examples/research-foundations.ipynb")
).resolve()


class NotebookDisplay:
    """Render small tables for the exploratory notebook scenarios."""

    @staticmethod
    def show_table(headers: tuple[str, ...], rows: list[tuple[object, ...]]) -> None:
        """Display a small escaped table without adding a dataframe dependency.

        Args:
            headers: Column labels.
            rows: Exact or presentation values to render as text.
        """
        heading = "".join(
            f'<th style="text-align:left;padding:6px">{escape(label)}</th>' for label in headers
        )
        body = "".join(
            "<tr>"
            + "".join(
                f'<td style="padding:6px;vertical-align:top">{escape(str(value))}</td>'
                for value in row
            )
            + "</tr>"
            for row in rows
        )
        display(HTML(f"<table><thead><tr>{heading}</tr></thead><tbody>{body}</tbody></table>"))

In [2]:
assert platform.python_version() == (root / ".python-version").read_text().strip() == "3.14.7"
installed = Path(alpha_research.__file__).resolve().parent
assert "site-packages" in installed.parts and installed != root / "src/alpha_research"
source_root = root / "src/alpha_research"
runtime_files = sorted(
    path
    for path in source_root.rglob("*")
    if path.is_file() and path.suffix in {".py", ".json", ".typed"}
)
assert runtime_files
runtime_digest = hashlib.sha256()
for source_file in runtime_files:
    relative = source_file.relative_to(source_root)
    source_bytes = source_file.read_bytes()
    assert source_bytes == (installed / relative).read_bytes(), f"Stale installed file: {relative}"
    runtime_digest.update(str(relative).encode() + b"\0" + source_bytes + b"\0")

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=root, text=True).strip()
branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=root, text=True).strip()
working_status = subprocess.check_output(["git", "status", "--short"], cwd=root, text=True).strip()
candidate_diff = subprocess.check_output(
    [
        "git",
        "diff",
        "--binary",
        "HEAD",
        "--",
        "src",
        "configs",
        "pyproject.toml",
        "uv.lock",
        ".python-version",
    ],
    cwd=root,
)
notebook_document = json.loads(notebook_path.read_text())
notebook_source = [
    (cell["cell_type"], "".join(cell["source"])) for cell in notebook_document["cells"]
]
notebook_digest = hashlib.sha256(
    json.dumps(notebook_source, ensure_ascii=False, separators=(",", ":")).encode()
).hexdigest()
identity = {
    "notebook_path": str(notebook_path),
    "attempt_id": os.environ.get("ALPHA_RESEARCH_ATTEMPT_ID", "interactive: unsnapshotted"),
    "commit": commit,
    "branch": branch,
    "working_tree": "changed" if working_status else "clean",
    "python": platform.python_version(),
    "platform": platform.platform(),
    "kernel": sys.executable,
    "package": str(installed),
    "verified_runtime_files": len(runtime_files),
    "runtime_sha256": runtime_digest.hexdigest(),
    "runtime_input_diff_sha256": hashlib.sha256(candidate_diff).hexdigest(),
    "uv_lock_sha256": hashlib.sha256((root / "uv.lock").read_bytes()).hexdigest(),
    "okx_config_sha256": hashlib.sha256(
        (root / "configs/history/okx-btc-usdt.json").read_bytes()
    ).hexdigest(),
    "notebook_source_sha256": notebook_digest,
    "notebook_tooling": {
        item.metadata["Name"]: version(item.metadata["Name"])
        for item in distributions()
        if item.metadata["Name"] in {"nbclient", "nbformat", "ipykernel"}
    },
}
NotebookDisplay.show_table(("Identity", "Observed value"), list(identity.items()))

Identity,Observed value
notebook_path,/Users/lxg/Documents/GitHub/Alpha Research/alpha-research-lab-private/examples/research-foundations.ipynb
attempt_id,06e4bde5-aa91-41a7-bb38-1d94aadfe4fb
commit,3deb34b966adf1f66e4eb79551375283c45300f0
branch,main
working_tree,changed
python,3.14.7
platform,macOS-27.0-arm64-arm-64bit-Mach-O
kernel,/Users/lxg/.cache/uv/builds-v0/.tmptYfVfU/bin/python
package,/Users/lxg/Documents/GitHub/Alpha Research/alpha-research-lab-private/.venv/lib/python3.14/site-packages/alpha_research
verified_runtime_files,72


## 2. Select the saved books

Pin `ALPHA_OKX_ARCHIVE_DIRECTORY` and `ALPHA_BITFINEX_CAPTURE_DIRECTORY` to repeat
an observation. Otherwise, this selects the latest completed input matching each
maintained configuration. The printed paths identify this convenience selection;
it is not yet a published research dataset. Invalid or absent inputs fail visibly.

In [3]:
codec = DomainCodec()
okx_config = OKXHistoryConfig.read(root / "configs/history/okx-btc-usdt.json")
bitfinex_config = CaptureConfig.read(root / "configs/live/bitfinex-btc-usd.json")
selected = {}
selection_rows = []
for venue, folder, variable, request_type, config in (
    ("okx", "data/okx/history", "ALPHA_OKX_ARCHIVE_DIRECTORY", OKXArchiveRequest, okx_config),
    (
        "bitfinex",
        "data/bitfinex",
        "ALPHA_BITFINEX_CAPTURE_DIRECTORY",
        CaptureRequest,
        bitfinex_config,
    ),
):
    override = os.environ.get(variable)
    candidates = []
    paths = (
        [Path(override).expanduser()]
        if override
        else [p.parent for p in (root / folder).glob("*/manifest.json")]
    )
    for path in paths:
        request = codec.decode_json((path / "request.json").read_bytes(), request_type)
        if request.config != config:
            if override:
                raise ValueError(f"{variable} differs from the maintained selection")
            continue
        descriptor = codec.decode_json((path / "manifest.json").read_bytes(), RawObject)
        candidates.append((descriptor.ts_ingested, str(path.resolve()), request, descriptor))
    if not candidates:
        raise FileNotFoundError(f"No matching {venue} input; run market-data-capture.ipynb first")
    _, location, request, descriptor = max(candidates, key=lambda item: (item[0], item[1]))
    selected[venue] = (Path(location), request, descriptor)
    selection_rows.extend(
        [
            (
                venue,
                "Selection",
                "explicit directory" if override else "latest matching saved input",
            ),
            (venue, "Directory", location),
            (venue, "Raw SHA-256", descriptor.sha256),
            (venue, "Request identity", request.identity),
            (venue, "Original implementation SHA-256", request.implementation_identity),
        ]
    )
NotebookDisplay.show_table(("Source", "Evidence", "Value"), selection_rows)

Source,Evidence,Value
okx,Selection,explicit directory
okx,Directory,/Users/lxg/Documents/GitHub/Alpha Research/alpha-research-lab-private/data/okx/history/802967bb-05eb-4b7b-9c90-2c964f22e41f
okx,Raw SHA-256,fef59b343d1972c32f67583edc538c5d2f7350e6f3a0059e6f87896d7e8bf676
okx,Request identity,71672efd00a8f3f6c959d0a08f8ce2a8793b3fb84d23f1a3c22b2756a6ec2665
okx,Original implementation SHA-256,22c0e38d88691bd0b1141588260465c471bbde0ff432a6bd1d0dab0505000014
bitfinex,Selection,explicit directory
bitfinex,Directory,/Users/lxg/Documents/GitHub/Alpha Research/alpha-research-lab-private/data/bitfinex/a5116c5a-b7d6-4e2b-9ee6-c79839c80f43
bitfinex,Raw SHA-256,a57ddabc5b1283c702ee2c7fd104a2b9cd943e2b41661cfc1d66619b35930163
bitfinex,Request identity,0e1453a2e7e925ad5bc1454a6a7d1e7298eca46e82a0d07b10e7ad8fc1c9ec21
bitfinex,Original implementation SHA-256,22c0e38d88691bd0b1141588260465c471bbde0ff432a6bd1d0dab0505000014


## 3. Reconstruct and inspect visible liquidity

**OKX BTC-USDT:** verify the historical archive for 8 September 2026 and reconstruct
its first **10,000 records**. L2 aggregates size at each price; it supplies no
individual order identity. This reader returns a summary, not a historical price table.

In [4]:
okx_directory, okx_request, okx_descriptor = selected["okx"]
history = OKXArchiveStore(okx_directory).replay()
assert history["source_sequence"] is None and history["gaps"] is None
epoch = datetime(1970, 1, 1, tzinfo=UTC)
history_rows = [
    (key, history[key])
    for key in (
        "symbol",
        "representation",
        "records",
        "snapshots",
        "updates",
        "levels",
        "depth",
        "state_available",
        "continuity",
        "stop_reason",
        "entire_archive_replayed",
    )
]
for key in ("first_source_ms", "last_source_ms"):
    value = history[key]
    history_rows.append(
        (
            key.replace("_ms", "_utc"),
            None if value is None else epoch + timedelta(milliseconds=int(value)),
        )
    )
NotebookDisplay.show_table(("Historical observation", "Result"), history_rows)

Historical observation,Result
symbol,BTC-USDT
representation,L2
records,10000
snapshots,1
updates,9999
levels,800
depth,400
state_available,True
continuity,unproven_no_source_sequences
stop_reason,record_limit


`gaps=None` means continuity is unknown because the archive has no source sequence
numbers. An available reconstructed state does not prove uninterrupted coverage.

**Bitfinex BTC/USD:** replay the saved bounded R0 recording, then inspect the nearest
three individual orders on each side. Prices and spreads are USD per BTC; sizes
are BTC. Configured depth 25 and the number of retained orders are different measures.
The spread describes this recording and does not account for execution costs.

In [5]:
bitfinex_directory, bitfinex_request, bitfinex_descriptor = selected["bitfinex"]
replay = CaptureReplay.read(bitfinex_directory)
summary = replay.summary()
assert isinstance(replay.book, BitfinexBook)
live_rows = [
    (key, summary[key])
    for key in (
        "symbol",
        "messages",
        "snapshots",
        "updates",
        "orders",
        "depth",
        "gaps",
        "synchronized",
        "source_sequence",
        "last_source_time",
    )
]
for key in ("started_ns", "finished_ns"):
    live_rows.append(
        (key.replace("_ns", "_utc"), epoch + timedelta(microseconds=summary[key] // 1000))
    )
NotebookDisplay.show_table(("Recorded observation", "Result"), live_rows)
assert summary["synchronized"] and not summary["gaps"], (
    "This sample is unsuitable for the uninterrupted-book preview; inspect its gaps"
)

preview_rows = []
best = {}
for side in ("buy", "sell"):
    orders = sorted(
        (order for order in replay.book.orders.values() if order.side == side),
        key=lambda order: order.price,
        reverse=side == "buy",
    )
    assert orders, f"No observed {side} side"
    best[side] = orders[0].price
    for order in orders[:3]:
        assert isinstance(order.price, Decimal) and isinstance(order.size, Decimal)
        preview_rows.append((side, order.order_id, str(order.price), str(order.size)))
NotebookDisplay.show_table(("Side", "Visible order ID", "USD / BTC", "BTC"), preview_rows)
NotebookDisplay.show_table(
    ("Measure", "USD / BTC"),
    [
        ("Best bid", best["buy"]),
        ("Best ask", best["sell"]),
        ("Observed spread", best["sell"] - best["buy"]),
    ],
)

Recorded observation,Result
symbol,tBTCUSD
messages,376
snapshots,1
updates,371
orders,50
depth,25
gaps,[]
synchronized,True
source_sequence,373
last_source_time,None


Side,Visible order ID,USD / BTC,BTC
buy,244232632883,75645,0.00180442
buy,244230114777,75639,0.0002
buy,244218842321,75638,0.0002
sell,244216587539,75655,0.00022
sell,244232116576,75660,0.03522185
sell,244217678512,75660,0.00022


Measure,USD / BTC
Best bid,75645
Best ask,75655
Observed spread,10


## 4. Check clocks and repeatability

OKX supplies historical event timestamps but no historical receipt clock. Bitfinex
R0 supplies local receipt order but no exchange timestamp in these messages. Their
quote currencies, times and depths differ; **USD is not USDT** and these are not
simultaneous price observations. UTC display truncates journal nanoseconds to microseconds.

Replaying the same bytes must reproduce the same state trace. The final control
injects a disconnect into a separate replay: the book becomes unavailable until
resynchronization, instead of carrying stale quotes across the gap.

In [6]:
NotebookDisplay.show_table(
    ("Sample", "Source clock", "Local receipt evidence", "Saved bytes"),
    [
        (
            "OKX historical BTC-USDT",
            "Unix milliseconds in records; continuity unknown",
            f"Archive received {history['download_received']}; per-record receipt absent",
            history["saved_bytes"],
        ),
        (
            "Bitfinex recorded BTC/USD",
            "Absent in R0 messages",
            f"Journal nanoseconds {summary['started_ns']} to {summary['finished_ns']}",
            bitfinex_descriptor.size_bytes,
        ),
    ],
)

again = CaptureReplay.read(bitfinex_directory)
assert again.summary()["trace_identity"] == summary["trace_identity"]
exercise = CaptureReplay(bitfinex_config)
for record in CaptureJournal(bitfinex_directory, bitfinex_request).records():
    if record.kind == "end":
        break
    exercise.apply(record)
exercise.apply(
    CaptureRecord(
        kind="gap",
        epoch=exercise.epoch,
        received_ns=exercise.finished_ns,
        reason="synthetic_disconnect",
    )
)
assert not exercise.book.ready
NotebookDisplay.show_table(
    ("Check", "Observed result"),
    [
        ("Repeated state trace", summary["trace_identity"]),
        ("Injected disconnect", f"Book usable: {exercise.book.ready}"),
    ],
)

Sample,Source clock,Local receipt evidence,Saved bytes
OKX historical BTC-USDT,Unix milliseconds in records; continuity unknown,Archive received 2026-09-16T08:49:14.475436+00:00; per-record receipt absent,109423187
Bitfinex recorded BTC/USD,Absent in R0 messages,Journal nanoseconds 1789548508888411000 to 1789548529076738000,52908


Check,Observed result
Repeated state trace,283533b937763c40d108517fc22ddbbc2d10a77820a8de498cc631f37c11d375
Injected disconnect,Book usable: False


## 5. Prepare separate inputs for one dataset

The archive workflow is:

**saved acquisitions + per-input evidence → admission and assembly → atomic publication → verified reading**

`DatasetInput` carries each acquisition's quality policy, eligibility history and
correction evidence. Assembly retains the source, stream and instrument of every
partition. It is the composition boundary for future combinations of LOB sources;
those streams will remain separate within the dataset.

Today the installed saved-input adapter supports trade archives. This offline
example creates two **synthetic** instrument inputs using that adapter, with an
intentional duplicate in the first input. The CSV construction below is fixture
setup; the publication calls use the generic API. Actual LOB publication waits for
MR 25. The real OKX and Bitfinex recordings above are not published here.

In [7]:
# standard library imports
from tempfile import TemporaryDirectory
from zipfile import ZIP_DEFLATED, ZipFile, ZipInfo

# third party library imports
from msgspec.structs import replace

# project library imports
from alpha_research.configuration import Configuration
from alpha_research.data.asof import AsOfSelector
from alpha_research.data.datasets import DatasetInput
from alpha_research.data.eligibility import EligibilityAssertions
from alpha_research.data.manifests import AcquisitionManifest
from alpha_research.data.quality import QualityPolicy
from alpha_research.data.raw_store import RawStore
from alpha_research.data.requests import SourceRequest
from alpha_research.data.snapshots import SnapshotStore
from alpha_research.pipelines.dataset import publish_dataset
from alpha_research.reference_data import CatalogSnapshot

archive_catalog = codec.decode_json(
    (root / "configs/reference/binance-archive-spot.json").read_bytes(), CatalogSnapshot
)
archive_run = Configuration.read(root / "configs/data-jobs/binance-archive.json").resolve(
    "btc-archive", archive_catalog
)
eligibility = EligibilityAssertions.read(
    root / "tests/fixtures/mr12/eligibility.json", catalog=archive_catalog
)
start = archive_run.job.start
policy = QualityPolicy(use="exploratory", known_at=start, max_gap=timedelta(hours=1))
max_bytes = 262_144
acquired_at = datetime(2026, 9, 17, tzinfo=UTC)
workspace = TemporaryDirectory(prefix="research-dataset-")
local = Path(workspace.name)
raw_directory, destination = local / "raw", local / "snapshots"
raw_directory.mkdir()
destination.mkdir()
store = RawStore(raw_directory)
snapshots = SnapshotStore(destination, store)
selections = []
for index, instrument in enumerate(archive_catalog.instruments):
    run = replace(archive_run, instruments=(instrument,))
    request = SourceRequest.partition(run)[0]
    source = local / "source.zip"
    member = ZipInfo(f"{instrument.symbol}-trades-{request.start.date()}.csv")
    member.compress_type = ZIP_DEFLATED
    with ZipFile(source, "w") as archive:
        archive.writestr(member, b"12,100.00,2.0,200,1717200000001,true,true\n" * (2 - index))
    origin = RawObject.from_file(
        source,
        request=request,
        retrieval_reference="synthetic:dataset-walkthrough",
        ts_received=acquired_at,
        ts_ingested=acquired_at,
    )
    store.publish(source, descriptor=origin, request=request)
    selections.append(
        DatasetInput(
            acquisition=AcquisitionManifest(run=run, completed=(origin,)),
            policy=replace(policy, max_gap=timedelta(minutes=60 - 30 * index)),
            eligibility=replace(
                eligibility,
                assertions=tuple(
                    row
                    for row in eligibility.assertions
                    if row.instrument_id == instrument.instrument_id
                ),
            ),
        )
    )
inputs = tuple(selections)

## 6. Publish, then reopen by identity

One `publish_dataset` call performs admission, assembly and atomic publication.
Its callbacks expose each quality report and publication progress. These sparse
archives remain **exploratory degraded**: known duplicates are retained as lineage,
while missing historical availability and coverage remain explicit limitations.

In [8]:
admission, progress = {}, {}
snapshot_id = publish_dataset(
    inputs,
    store,
    destination,
    max_bytes=max_bytes,
    report=admission.__setitem__,
    progress=progress.__setitem__,
)
assert tuple(progress) == ("started", "complete")
assert all(manifest.identity == snapshot_id for manifest in progress.values())
assert [outcome.duplicate_count for outcome in admission.values()] == [1, 0]
NotebookDisplay.show_table(
    ("Input", "Source / stream", "Scope", "Invalid / gaps / duplicates"),
    [
        (
            selected.acquisition.run.instruments[0].symbol,
            (selected.acquisition.run.instruments[0].source, selected.acquisition.run.job.stream),
            outcome.scope,
            (outcome.invalid_count, outcome.gap_count, outcome.duplicate_count),
        )
        for selected, outcome in zip(inputs, admission.values(), strict=True)
    ],
)
print("Published snapshot:", snapshot_id)
print("Retained limitations:", "; ".join(next(iter(admission.values())).reasons))

Input,Source / stream,Scope,Invalid / gaps / duplicates
BTCUSDT,"('binance.archive.spot', 'trade')",exploratory_degraded,"(0, 0, 1)"
ETHUSDT,"('binance.archive.spot', 'trade')",exploratory_degraded,"(0, 1, 0)"


Published snapshot: 2b6e57d340056488a630b77ae1ffccce405addea22b1b34289ef2c8a5a3592d3
Retained limitations: catalog units are authored research assumptions; original trade availability unknown; recorder coverage unknown


In [9]:
reopened = snapshots.read(snapshot_id, max_bytes=max_bytes)
assert len(reopened.partitions) == 2
assert tuple(part.policy for part in reopened.partitions) == tuple(item.policy for item in inputs)
assert tuple(part.eligibility for part in reopened.partitions) == tuple(
    item.eligibility for item in inputs
)
# Exact repeated selections are reused; the two instrument bindings remain separate.
assert publish_dataset((*inputs, inputs[0]), store, destination, max_bytes=max_bytes) == snapshot_id
precedence = tuple(
    (
        item.acquisition.run.instruments[0].source,
        item.acquisition.run.job.stream,
        item.acquisition.run.instruments[0].instrument_id,
    )
    for item in inputs
)
reader = AsOfSelector.read(
    destination, store, identity=snapshot_id, max_bytes=max_bytes, precedence=precedence
)
selected_events = reader.select(
    tuple(binding[2] for binding in precedence),
    start=start,
    end=archive_run.job.end,
    known_at=start + timedelta(seconds=1),
)
assert selected_events == ()
assert all(b"original trade availability unknown" in payload for payload in reader.history)
NotebookDisplay.show_table(
    ("Reopened input", "Canonical SHA-256", "Gap tolerance"),
    [
        (part.acquisition.run.instruments[0].symbol, part.sha256, part.policy.max_gap)
        for part in reopened.partitions
    ],
)
print("Known archive events selected:", len(selected_events))

Reopened input,Canonical SHA-256,Gap tolerance
BTCUSDT,021fe1e5092daf332472d0c579f1aa3e628a80d0e799b1c710be608d7c16814e,1:00:00
ETHUSDT,f46235da0a4150eb512b8cd9bf8fbec4cc8b0177eb3f0ffb92c031d43999fa70,0:30:00


Known archive events selected: 0


The explicit snapshot ID reopens the raw bytes, canonical partitions and retained
evidence. An exact repeat yields the same identity. Both instrument partitions
survive even though their synthetic trade IDs match.

The cutoff query selects **zero** archive events because their original availability
is unknown. Download time does not establish historical knowledge, and an empty
selection does not prove a quiet or complete market. Full histories remain in
`reader.history`; those bytes include later information and are not a cutoff view.

### The same workflow from the command line

Serialize the same `DatasetInput` records and pass them to `alpha-dataset publish
--inputs`. This invocation runs outside the checkout and must return the same ID
as Python. For operating retained stores, see the [dataset API guide](../src/alpha_research/data/README.md).

In [10]:
# standard library imports
from shutil import which

executable = which("alpha-dataset")
assert executable is not None, "Refresh the installed package first."
input_paths = [local / f"input-{index}.json" for index in range(len(inputs))]
for path, item in zip(input_paths, inputs, strict=True):
    path.write_bytes(codec.encode_json(item))
command = subprocess.run(
    [
        executable,
        "publish",
        "--store",
        str(raw_directory),
        "--snapshots",
        str(destination),
        "--max-bytes",
        str(max_bytes),
        "--inputs",
        *(str(path) for path in input_paths),
    ],
    cwd=local,
    capture_output=True,
    text=True,
    check=False,
    timeout=30,
)
assert command.returncode == 0, (command.stdout, command.stderr)
records = [json.loads(line) for line in command.stdout.splitlines()]
assert records[-1]["status"] == "complete" and records[-1]["snapshot_id"] == snapshot_id
NotebookDisplay.show_table(
    ("Stage", "Status", "Scope"),
    [(row["stage"], row["status"], row.get("scope", "")) for row in records],
)

Stage,Status,Scope
configuration,started,
normalize,started,
normalize,validated,exploratory_degraded
normalize,validated,exploratory_degraded
publish,started,
publish,persisted,
publish,complete,exploratory_degraded


### Exercise: what changes a dataset's identity?

Predict the effect of tightening only the first input's gap tolerance from one hour
to five minutes. Should its canonical partition change? Should the second partition
or the earlier snapshot change? Run the next cell to check your answer.

In [11]:
changed_inputs = (
    replace(inputs[0], policy=replace(inputs[0].policy, max_gap=timedelta(minutes=5))),
    inputs[1],
)
changed_id = publish_dataset(changed_inputs, store, destination, max_bytes=max_bytes)
changed = snapshots.read(changed_id, max_bytes=max_bytes)
assert changed_id != snapshot_id
assert changed.partitions[0].sha256 != reopened.partitions[0].sha256
assert changed.partitions[1] == reopened.partitions[1]
assert snapshots.read(snapshot_id, max_bytes=max_bytes) == reopened
NotebookDisplay.show_table(
    ("Comparison", "Outcome"),
    [
        ("Dataset and first partition", "New identities"),
        ("Second partition", "Unchanged"),
        ("Earlier snapshot", "Still readable by its ID"),
    ],
)

Comparison,Outcome
Dataset and first partition,New identities
Second partition,Unchanged
Earlier snapshot,Still readable by its ID


The answer: policy is part of the evidence, so changing it changes identity; the
other input and earlier snapshot remain intact. Two representative failures below
show the boundary: causal use cannot accept unknown required coverage, and readers
require an immutable ID. Detailed corruption and interruption cases live in the
[dataset tests](../tests/unit/data/test_datasets.py),
[snapshot tests](../tests/unit/data/test_snapshots.py) and
[command tests](../tests/unit/pipelines/test_dataset_command.py).

In [12]:
rejections = []
try:
    publish_dataset(
        (replace(inputs[0], policy=replace(policy, use="causal")),),
        store,
        destination,
        max_bytes=max_bytes,
    )
except ValueError as error:
    rejections.append(("Causal publication", str(error)))
else:
    raise AssertionError("Accepted unknown required coverage")
try:
    snapshots.read("latest", max_bytes=max_bytes)
except ValueError as error:
    rejections.append(("Mutable snapshot alias", str(error)))
else:
    raise AssertionError("Accepted mutable snapshot alias")
NotebookDisplay.show_table(("Refused request", "Reason"), rejections)

Refused request,Reason
Causal publication,partition publication refused: catalog units are authored research assumptions; original trade availability unknown; recorder coverage unknown
Mutable snapshot alias,snapshot requires an explicit lowercase SHA-256 identity


## 7. Select what was known at a cutoff

To see a nonempty cutoff selection, use a **synthetic captured trade** with an
original receipt clock, plus an explicitly evidenced correction. This fixture is
admitted in memory; a persisted captured-input adapter is not implemented here.

The correction is published at second 9 and available at second 10. A separate
synthetic membership correction becomes known at noon and withdraws the earlier
listing. Both histories are retained. This demonstrates the MR 09–17 semantics
without assigning invented receipt times to the archive example.

In [13]:
# project library imports
from alpha_research.data.normalization import Occurrence
from alpha_research.data.revisions import Correction, TradeRevision
from alpha_research.pipelines.dataset import inspect_fixture

captured_catalog = codec.decode_json(
    (root / "configs/reference/binance-spot.json").read_bytes(), CatalogSnapshot
)
captured_run = Configuration.read(root / "configs/data-jobs/fixture-inspection.json").resolve(
    "btc-fixture", captured_catalog
)
original_line = (root / "tests/fixtures/m0/btcusdt.txt").read_bytes()
versions = []
for raw in (original_line, original_line.replace(b'"p":"67540.01000000"', b'"p":"67541.01000000"')):
    frame = local / "frame.txt"
    frame.write_bytes(raw)
    binding, event, digest = inspect_fixture(
        captured_run,
        frame,
        local_received=acquired_at,
        local_ingested=acquired_at,
        source_sequence=0,
    )
    origin = RawObject.from_file(
        frame,
        request=binding,
        retrieval_reference="synthetic:cutoff-walkthrough",
        ts_received=acquired_at,
        ts_ingested=acquired_at,
    )
    assert digest == origin.sha256
    versions.append(Occurrence(request=binding, origin=origin, position=0, raw=raw, event=event))
original_revision = TradeRevision(occurrences=(versions[0],))
correction = Correction(
    supersedes=original_revision.identity,
    replacement=TradeRevision(occurrences=(versions[1],)).identity,
    ts_published=start + timedelta(seconds=9),
    ts_available=start + timedelta(seconds=10),
    evidence_reference="synthetic:cutoff-walkthrough:correction",
)
captured_eligibility = replace(eligibility, catalog=captured_catalog)
report = policy.validate(
    binding,
    versions,
    eligibility=captured_eligibility,
    corrections=(correction,),
)
selector = AsOfSelector(
    (report.require_canonical(),),
    precedence=(
        (binding.instrument.source, binding.manifest.job.stream, binding.instrument.instrument_id),
    ),
)

In [14]:
cutoffs = (
    start,
    start + timedelta(seconds=1),
    correction.ts_published,
    correction.ts_available,
    start + timedelta(hours=12),
)
cutoff_rows = []
for cutoff in cutoffs:
    visible = selector.select(
        (binding.instrument.instrument_id,),
        start=start,
        end=binding.end,
        known_at=cutoff,
    )
    cutoff_rows.append(
        (
            cutoff,
            [str(row.occurrences[0].event.price) for row in visible],
            [row.ts_knowledge for row in visible],
        )
    )
assert [row[1] for row in cutoff_rows] == [
    [],
    ["67540.01000000"],
    ["67540.01000000"],
    ["67541.01000000"],
    [],
]
assert selector.select(
    (binding.instrument.instrument_id,),
    start=start,
    end=binding.end,
    known_at=start + timedelta(seconds=1),
) == (original_revision,)
assert selector.history == (report.require_canonical(),)
NotebookDisplay.show_table(
    ("Known at", "Selected price", "Effective knowledge"),
    [
        (
            cutoff.isoformat(),
            ", ".join(prices) or "Unavailable",
            ", ".join(clock.isoformat() for clock in clocks) or "—",
        )
        for cutoff, prices, clocks in cutoff_rows
    ],
)
print("Retained quality scope:", report.scope)

Known at,Selected price,Effective knowledge
2024-06-01T00:00:00+00:00,Unavailable,—
2024-06-01T00:00:01+00:00,67540.01000000,2024-06-01T00:00:00.004103+00:00
2024-06-01T00:00:09+00:00,67540.01000000,2024-06-01T00:00:00.004103+00:00
2024-06-01T00:00:10+00:00,67541.01000000,2024-06-01T00:00:10+00:00
2024-06-01T12:00:00+00:00,Unavailable,—


Retained quality scope: exploratory_degraded


The old price remains visible at second 9; the replacement enters at second 10.
At noon, the newly known membership correction removes the event from the selected
universe. Repeating the earlier query still returns the original revision. Validity
and knowledge are separate clocks; using the final revision everywhere would leak
future information into earlier decisions.

This is a bounded reference selector over individual trades. It establishes
software behavior, not empirical historical eligibility or complete source coverage.
More cases—equal-time ordering, binding-scoped disconnects and invalid queries—are
retained in the [as-of tests](../tests/unit/data/test_asof.py) and
[installed consumer](../tests/fixtures/mr17/asof_consumer.py). Historical NB 09–17 evidence
remains in the [MR records](../docs/roadmap/README.md).

For a new book sample use [capture](market-data-capture.ipynb); for unit, timing and
storage failure experiments use the [offline lab](foundation-contract-checks.ipynb).
The next dataset work is bounded reading in MR 19 and LOB adapters in MR 25.

In [15]:
# All synthetic files belong to this run; the saved book inputs were only read.
workspace.cleanup()